<a href="https://colab.research.google.com/github/Mohan102945/mohan102945.github.io/blob/main/LLM_From_scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import urllib.request

url = ("https://raw.githubusercontent.com/rasbt/"
       "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
       "the-verdict.txt")

file_path = "the-verdict.txt"

urllib.request.urlretrieve(url, file_path)

with open(file_path, 'r') as file:
    raw_text = file.read()

In [4]:
print(type(raw_text))

<class 'str'>


In [5]:
print(raw_text[:50])

I HAD always thought Jack Gisburn rather a cheap g


In [11]:
type(token_ids)

list

In [10]:
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")
token_ids = tokenizer.encode(raw_text)

In [8]:
import torch
from torch.utils.data import Dataset, DataLoader

In [36]:
class GPTDatasetv1(Dataset):
  def __init__(self,text,tokenizer,max_length,stride):
    self.input_ids = []
    self.output_ids = []

    token = tokenizer.encode(text)

    for i in range(0,len(token)-max_length,stride):
       input_chunk = token[i:i+max_length]
       output_chunk = token[i+1:i+max_length+1]
       self.input_ids.append(torch.tensor(input_chunk))
       self.output_ids.append(torch.tensor(output_chunk))

  def __len__(self):
      return len(self.input_ids)

  def __getitem__(self,idx):
      return self.input_ids[idx],self.output_ids[idx]

In [37]:
def create_dataloader_v1(text, batch_size, max_length, stride, shuffle, drop_last, num_workers):
    tokenizer = tiktoken.get_encoding("gpt2")

    dataset = GPTDatasetv1(text, tokenizer, max_length, stride)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, drop_last=drop_last, num_workers=num_workers)


    return dataloader

In [39]:
batch_size = 8
max_length = 8
stride = 1
shuffle = True
drop_last = True
num_workers = 0

dataloader = create_dataloader_v1(raw_text,batch_size = batch_size,max_length = max_length, stride = stride, shuffle = shuffle, drop_last = drop_last, num_workers = num_workers)
data_iter = iter(dataloader)

inputs,target = next(data_iter)
print(inputs)


tensor([[   30,  2011, 20136,   373,  3957,   588,   262, 26394],
        [  536,  5469,   550, 11001,    11,   262,  2756,   286],
        [ 1549,  1239, 12615,   257, 14093,   526,   198,   198],
        [ 1359,    26,   788,   314,  3114,   379,   262, 50085],
        [  373,   262,   705,  4976,     6,   582,    11,   290],
        [ 8631,  2119,   379,   262,   886,   286,   262,   781],
        [   11,   339,   550,  5710,   465, 12036,    11,  6405],
        [  345,   287,  1936,  2431,   438,   392,   340,  1422]])


In [41]:
embedding_layer = torch.nn.Embedding(num_embeddings=50257, embedding_dim=256)
embedding = embedding_layer(inputs)
embedding.shape

torch.Size([8, 8, 256])

In [42]:
pos_embedding_layer = torch.nn.Embedding(num_embeddings = max_length, embedding_dim = 256)
pos_embedding = pos_embedding_layer(torch.arange(max_length))
pos_embedding.shape

torch.Size([8, 256])

In [43]:
input = embedding + pos_embedding
input.shape

torch.Size([8, 8, 256])